# Get individual pages from iiif manifest

In [12]:
import pandas as pd
import json
import requests
from tqdm.notebook import trange, tqdm
import time
import shutil
import os
from io import BytesIO
import re
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [13]:
def make_session():
    session = requests.Session()
    retry = Retry(total=5, backoff_factor=1.0, status_forcelist=[429, 500, 502, 503, 504], allowed_methods=["GET"], respect_retry_after_header=True)
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    session.headers.update({"User-Agent": "research-script/1.0 (DHInfra Uni Graz)"})
    return session

In [3]:
iiif_df = pd.read_csv("data_dh-infra.csv")

In [16]:
session = make_session()

images_list = []
id_list = []
title_list = []

DELAY = 0.5 

for index, row in tqdm(iiif_df.iterrows(), total=len(iiif_df)):
    url = row['iiif_manifest']
    try:
        response = session.get(url, timeout=30)
        response.raise_for_status()
        data = response.json()

        for entry in data.get("metadata", []):
            label_de = entry.get("label", {}).get("de", [""])[0]
            if label_de == "Jahr/Datierung":
                edition_date = entry.get("value", {}).get("de", [""])[0]
                edition_year = pd.to_datetime(edition_date, dayfirst=True).strftime('%Y')
                edition_date = pd.to_datetime(edition_date, dayfirst=True).strftime('%d-%m-%Y')
                break

        images = [anno["body"]["id"] for canvas in data.get("items", []) for page in canvas.get("items", []) for anno in page.get("items", []) 
                  if anno.get("motivation") == "painting"]
        images_id = [f"{row['anno_id']}_{i}" for i in range(len(images))]
        images_list.extend(images)
        id_list.extend(images_id)
        title_list.extend([row['title']] * len(images))

    except requests.exceptions.RequestException as e:
        print(f"Error fetching the URL: {e}")
    except Exception as e:
        print(f"An error occurred: {e}")

    time.sleep(DELAY)

  0%|          | 0/16791 [00:00<?, ?it/s]

In [17]:
page_df = pd.DataFrame({'title': title_list, 'anno_id': id_list, 'iiif_manifest': images_list})

In [22]:
page_df[['short_title', 'year']] = page_df['anno_id'].str.extract(r'^([a-zA-Z]+)(\d{4})')

In [24]:
page_df.to_csv("data_dh-infra_pages.csv", index=False)

# Layout Analysis

## Enyollah (best for complex pages)

In [6]:
import subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

In [2]:
def run_year(short_title, year, file_list, gpu_id):
    log_path = LOG_DIR / f"{short_title}_{year}.log"
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    env["CONDA_PREFIX"] = str(EYNOLLAH_PREFIX)
    env["PATH"] = f"{EYNOLLAH_PREFIX}/bin:" + env.get("PATH", "")

    with open(log_path, "w") as logf:
        logf.write(f"[orchestrator] pinned to GPU {gpu_id}\n")
        logf.flush()
        result = subprocess.run(["bash", str(SCRIPT), file_list, short_title, str(year), str(WORK_DIR)], stdout=logf, stderr=subprocess.STDOUT,
                                env=env)
    return (short_title, year), result.returncode

In [3]:
def is_complete(short_title, year, list_file):
    out_dir = LAYOUT_DIR / short_title / str(year)
    if not out_dir.exists():
        return False
    n_inputs = sum(1 for _ in open(list_file))
    n_outputs = len(list(out_dir.glob("*.xml")))
    return n_outputs >= n_inputs

In [4]:
EYNOLLAH_PREFIX = Path("/home/lab6/miniforge3/envs/eyn_env") # absolute path of env containing Eynollah
SCRIPT = Path("eynollah_anno.sh").resolve()
WORK_DIR = Path.cwd()
# adjust N_GPUS and JOBS_PER_GPU based on available ressources
# one session takes around 13GB VRAM and 20 to 40 GB RAM  
N_GPUS = 2 
JOBS_PER_GPU = 2
MAX_PARALLEL = N_GPUS * JOBS_PER_GPU

ANNO_SOURCE = WORK_DIR / "anno_source"
LAYOUT_DIR = WORK_DIR / "Layout"
LOG_DIR = WORK_DIR / "logs"

for d in (ANNO_SOURCE, LAYOUT_DIR, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)

In [40]:
year_jobs = []  # list of (short_title, year, list_file_path)

for (short_title, year), group in page_df.groupby(["short_title", "year"]):
    out_dir = ANNO_SOURCE / short_title
    out_dir.mkdir(parents=True, exist_ok=True)
    list_file = out_dir / f"{year}.txt"

    with open(list_file, "w") as f:
        for _, row in group.iterrows():
            f.write(f"{row['anno_id']}\t{row['iiif_manifest']}\n")

    year_jobs.append((short_title, int(year), str(list_file)))

print(f"Wrote {len(year_jobs)} file lists under {ANNO_SOURCE}")

Wrote 57 file lists under /home/lab6/job_ads/anno_source


In [41]:
pending_jobs = [(st, yr, fl) for st, yr, fl in year_jobs if not is_complete(st, yr, fl)]
print(f"{len(pending_jobs)} of {len(year_jobs)} jobs pending " f"({len(year_jobs) - len(pending_jobs)} already complete)")

57 of 57 jobs pending (0 already complete)


In [50]:
with ThreadPoolExecutor(max_workers=MAX_PARALLEL) as executor:
    futures = {executor.submit(run_year, st, yr, fl, i % N_GPUS): (st, yr) for i, (st, yr, fl) in enumerate(pending_jobs)}

    failed = []
    with tqdm(total=len(futures), desc="Paper-year batches") as pbar:
        for future in as_completed(futures):
            (st, yr), rc = future.result()
            if rc != 0:
                failed.append((st, yr))
                pbar.write(f"✗ {st}/{yr} — see {LOG_DIR}/{st}_{yr}.log")
            else:
                pbar.write(f"✓ {st}/{yr}")
            pbar.update(1)
if failed:
    print(f"\n{len(failed)} job(s) failed:")
    for st, yr in failed:
        print(f"  {st}/{yr}  →  {LOG_DIR}/{st}_{yr}.log")
else:
    print("\nAll jobs completed successfully.")

Paper-year batches:   0%|          | 0/4 [00:00<?, ?it/s]

✓ guv/1891
✓ guv/1894
✓ guv/1893
✓ guv/1892

All jobs completed successfully.


# OCR

## Adjust filepath in XML-File to IIIF Link

In [54]:
import os
import time
from pathlib import Path
from tqdm.auto import tqdm
from ocrd_models.ocrd_page import parse, to_xml

In [55]:
def process_single_file(xml_file, lookup):
    try:
        pcgts = parse(xml_file, silence=True)
        page = pcgts.get_Page()
        old_filename = page.get_imageFilename()

        # already converted to a URL skip
        if str(old_filename).startswith("http"):
            return ("skipped", xml_file, "already a URL")

        anno_id = Path(old_filename).stem          
        new_filename = lookup.get(anno_id)
        if new_filename is None:
            return ("error", xml_file, f"anno_id not in df: {anno_id}")

        page.set_imageFilename(new_filename)
        tmp = xml_file + ".tmp"
        with open(tmp, "w", encoding="utf-8") as f:
            f.write(to_xml(pcgts))
        os.replace(tmp, xml_file)
        return ("success", xml_file, None)
    except Exception as e:
        return ("error", xml_file, f"{type(e).__name__}: {e}")

In [56]:
url_lookup = dict(zip(page_df["anno_id"], page_df["iiif_manifest"]))

In [57]:
input_root = "/home/lab6/job_ads/Layout/"
xml_files = []
with tqdm(desc="Scanning input", unit=" file") as pbar:
    for root, _, files in os.walk(input_root):
        for f in files:
            if f.endswith(".xml"):
                xml_files.append(os.path.join(root, f))
                pbar.update(1)

Scanning input: 0 file [00:00, ? file/s]

In [59]:
print(f"Processing {len(xml_files)} files...")
results = {"success": 0, "skipped": 0, "error": 0}
errors = []
for xml_file in tqdm(xml_files, desc="Updating XMLs"):
    status, path, msg = process_single_file(xml_file, url_lookup)
    results[status] += 1
    if status == "error":
        errors.append((path, msg))

print("-" * 30)
print(f"Success: {results['success']} | Skipped: {results['skipped']} | Errors: {results['error']}")
if errors:
    print(f"\nFirst 10 errors:")
    for path, msg in errors[:10]:
        print(f"  {path}: {msg}")

Processing 612 files...


Updating XMLs:   0%|          | 0/612 [00:00<?, ?it/s]

------------------------------
Success: 612 | Skipped: 0 | Errors: 0


## OCR

In [63]:
import os
from pathlib import Path
from tqdm.auto import tqdm
from ocr_pipeline_anno import run_pipeline, filter_unprocessed

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [61]:
input_root  = "Layout" 
output_root = "OCR"

xml_files = []
with tqdm(desc="Scanning XMLs", unit=" file") as pbar:
    for root, _, files in os.walk(input_root):
        for f in files:
            if f.endswith(".xml"):
                xml_files.append(os.path.join(root, f))
                pbar.update(1)
print(f"Found {len(xml_files)} XML files")

Scanning XMLs: 0 file [00:00, ? file/s]

Found 612 XML files


In [62]:
todo = filter_unprocessed(xml_files, input_root, output_root)
print(f"{len(todo)} of {len(xml_files)} still to process")

Checking resume:   0%|          | 0/612 [00:00<?, ? file/s]

612 of 612 still to process


In [64]:
errors = run_pipeline(todo, input_root = input_root, output_root = output_root, tessdata_path = "/usr/share/tesseract-ocr/5/tessdata", 
                      model_name = "frak2021-0.905", ocr_workers = 100, download_workers = 8, max_in_flight = 300, request_timeout  = 60.0)

Scratch: /dev/shm/ocr_iiif_n59bea1a


OCR pipeline:   0%|          | 0/612 [00:00<?, ? file/s]

All files processed successfully.


# Post Correction 

## Extracting Text from XML-Files

In [99]:
from ocrd_models.ocrd_page import parse
import pandas as pd
from concurrent.futures import ProcessPoolExecutor
from tqdm.auto import tqdm
import os
from pathlib import Path

In [97]:
def extract_text_in_reading_order(xml_path):
    pcgts = parse(xml_path, silence=True)
    page = pcgts.get_Page()
    regions = {r.id: r for r in page.get_AllRegions(classes=["Text"])}
    ordered_ids = _region_ids_in_reading_order(page, regions)
    result = []
    for rid in ordered_ids:
        region = regions.get(rid)
        if region is None:
            continue
        lines = []
        for line in region.get_TextLine():
            equivs = line.get_TextEquiv()
            if equivs and equivs[0].Unicode:
                lines.append(equivs[0].Unicode)
        if lines:
            result.append({"region_id": rid, "lines": lines, "text": "\n".join(lines)})
    return result


def page_text(xml_path, region_sep="\n\n"):
    return region_sep.join(r["text"] for r in extract_text_in_reading_order(xml_path))


def _region_ids_in_reading_order(page, regions):
    ro = page.get_ReadingOrder()
    if ro is None:
        return list(regions.keys())
    root = ro.get_OrderedGroup() or ro.get_UnorderedGroup()
    if root is None:
        return list(regions.keys())
    ordered = []
    _walk_group(root, ordered)
    mentioned = set(ordered)
    for rid in regions:
        if rid not in mentioned:
            ordered.append(rid)
    return ordered


def _walk_group(group, out):
    def safe(name):
        return getattr(group, name, lambda: None)() or []
    items = []
    for ref in safe("get_RegionRefIndexed"):
        items.append((ref.index, "ref", ref.regionRef))
    for ref in safe("get_RegionRef"):
        items.append((None, "ref", ref.regionRef))
    for sub in safe("get_OrderedGroupIndexed"):
        items.append((sub.index, "group", sub))
    for sub in safe("get_UnorderedGroupIndexed"):
        items.append((sub.index, "group", sub))
    for sub in safe("get_OrderedGroup"):
        items.append((None, "group", sub))
    for sub in safe("get_UnorderedGroup"):
        items.append((None, "group", sub))
    if any(idx is not None for idx, _, _ in items):
        items.sort(key=lambda x: (x[0] is None, x[0]))
    for _, kind, payload in items:
        if kind == "ref":
            out.append(payload)
        else:
            _walk_group(payload, out)

In [98]:
def _extract_rows(xml_path):
    """Returns a list of dicts, one per region."""
    try:
        regions = extract_text_in_reading_order(xml_path)
    except Exception as e:
        return [{"xml_path": xml_path,  "region_id": None, "reading_order": None, "text": None, "error": repr(e)}]
    return [{"xml_path": xml_path, "region_id": r["region_id"], "reading_order": i, "text": r["text"], "error": None} for i, r in enumerate(regions)]

def build_region_dataframe(xml_files, workers=32):
    """One row per region across all XMLs. Returns a DataFrame."""
    all_rows = []
    with ProcessPoolExecutor(max_workers=workers) as pool:
        for rows in tqdm(pool.map(_extract_rows, xml_files), total=len(xml_files), desc="Extracting regions"):
            all_rows.extend(rows)
    return pd.DataFrame(all_rows)

In [93]:
input_root  = "OCR"

xml_files = []
with tqdm(desc="Scanning XMLs", unit=" file") as pbar:
    for root, _, files in os.walk(input_root):
        for f in files:
            if f.endswith(".xml"):
                xml_files.append(os.path.join(root, f))
                pbar.update(1)
print(f"Found {len(xml_files)} XML files")

Scanning XMLs: 0 file [00:00, ? file/s]

Found 612 XML files


In [124]:
df = build_region_dataframe(xml_files, workers=32)

Extracting regions:   0%|          | 0/612 [00:00<?, ?it/s]

In [125]:
df["newspaper"] = df["xml_path"].apply(lambda p: Path(p).parts[-3])
df["year"] = df["xml_path"].apply(lambda p: Path(p).parts[-2])
df["page"] = df["xml_path"].apply(lambda p: Path(p).stem)

## Post Correction with hmByT5

In [ ]:
# The currently used model is not recommended due to inconsistent performance on OCR
# extracted from layout segmentation performed on full pages. 

In [115]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import numpy as np
from tqdm.auto import tqdm
from correction_multigpu import post_correct_multi_gpu

In [105]:
def byt5_byte_len(text):
    if not text:
        return 1
    return len(text.encode("utf-8")) + 1


def split_for_byt5(text, max_len=150, _seps=("\n\n", "\n", ". ", "; ", ", ", " ")):
    """
    Recursively split text so each chunk's ByT5 input length is <= max_len.
    Tries higher-quality separators first (paragraph -> sentence -> clause -> word),
    falls back to a UTF-8-safe byte-level split only for unbreakable runs.
    """
    if not text:
        return [text]
    if byt5_byte_len(text) <= max_len:
        return [text]

    # Try each separator in priority order; pick the first that yields a real split
    for sep in _seps:
        if sep not in text:
            continue
        parts = text.split(sep)
        if len(parts) < 2:
            continue
        mid = len(parts) // 2
        left = sep.join(parts[:mid])
        right = sep.join(parts[mid:])
        if left and right:
            return (split_for_byt5(left,  max_len, _seps)
                  + split_for_byt5(right, max_len, _seps))

    # No usable separator (single very long token): split at a UTF-8 char boundary
    encoded = text.encode("utf-8")
    mid = len(encoded) // 2
    # Walk forward past UTF-8 continuation bytes (0b10xxxxxx)
    while mid < len(encoded) and (encoded[mid] & 0xC0) == 0x80:
        mid += 1
    left  = encoded[:mid].decode("utf-8", errors="ignore")
    right = encoded[mid:].decode("utf-8", errors="ignore")
    return split_for_byt5(left, max_len, _seps) + split_for_byt5(right, max_len, _seps)

In [126]:
df['text'] = df['text'].str.replace(r'ﬁ', r'fi', regex=True)
df['text'] = df['text'].str.replace(r'ﬂ', r'fl', regex=True)
df['text'] = df['text'].str.replace(r'¬', r'-', regex=True)
df['text'] = df['text'].str.replace(r'¶', r'', regex=True)
df['text'] = df['text'].str.replace(r'˖', r'+', regex=True)
df['text'] = df['text'].str.replace(r'‟', r'"', regex=True)
df['text'] = df['text'].str.replace(r'⅙', r'1/6', regex=True)
df['text'] = df['text'].str.replace(r'ﬀ', r'ff', regex=True)
df['text'] = df['text'].str.replace(r'ꝓ', r'ff', regex=True)
df['text'] = df['text'].str.replace(r'ů', r'u', regex=True)
df['text'] = df['text'].str.replace(r'̃', r'', regex=True)
df['text'] = df['text'].str.replace(r'̈', r'', regex=True)
df['text'] = df['text'].str.replace(r'ﬄ', r'ffl', regex=True)
df['text'] = df['text'].str.replace(r'⸗', r'-', regex=True)
df['text'] = df['text'].str.replace(r'ꝛ', r'r', regex=True)
for vok, uml in [('a', 'ä'), ('o', 'ö'), ('u', 'ü')]:
    df['text'] = df['text'].str.replace(vok+r'ͤ', uml, regex=True)

In [127]:
df["chunks"] = df["text"].fillna("").map(lambda t: split_for_byt5(t, max_len=150))
df_split = (df.explode("chunks", ignore_index=True).rename(columns={"chunks": "split_text"}))
df_split["text_byt_len"] = df_split["split_text"].map(byt5_byte_len)

assert df_split["text_byt_len"].max() <= 150

In [130]:
corrected = post_correct_multi_gpu(df_split["split_text"].tolist(), model_path="Var3n/hmbyt5_frak_correction_100_adafactor_150", gpu_ids=[0, 1],
                                   batch_size=250, max_input_len=150, num_beams=4, dtype="bfloat16")
df_split["post_corrected"] = corrected

In [131]:
per_region = (df_split.groupby(["xml_path", "region_id"], sort=False)["post_corrected"].agg(" ".join).rename("post_corrected"))
df = df.merge(per_region, on=["xml_path", "region_id"], how="left")

In [134]:
#df.to_parquet("data_dh-infra_postcorrected.parquet", compression="zstd")
df.to_csv("data_dh-infra_postcorrected.csv", index=False)